# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


(Make header on home computer) Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [2]:
!pip install python-dotenv

import os
import json
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

## The question

Power creep is defined as "the strengthening of [a] game and its pieces over time possibly to the point where new pieces invalidate older ones" (citation). This occurs because developers want to make their games more exciting and fresh (will be continued later when I ask about where something like this would go)

Question: Has Pokemon experienced power creep over its thirty years of existance? If so, to what extent is power creep experienced?

## Methodology

### How will power creep be measured?

Pokemon is a complex game with many avenues to measure power creep: the functionality of newly-introduced abilities, the power and function of newly-introduced moves, and the type and amount of moves a Pokemon can learn, to name a few. My chosen unit of measurement for power creep is the average Base Stat Total (BST) across the nine generations of Pokemon. I have chosen this measure because though BSTs do not represent the actual stats of a Pokemon in game, those being affected by natures, Internal Values (IVs) and Effort Values (EVs), they represent the potential of how powerful a Pokemon could possibly be. Unchanged by outside factors, like abilities or type match-ups, the Pokemon with the higher stats is more likely to win a battle. Additionally, as a purely quantitative measure, it is easy to compare BSTs and understand immediately if one is higher or lower than another. They provide a simple way of measuring if Pokemon have overall been getting stronger through generations.

However, because Pokemon is such a complex game, solely looking at BSTs might not tell the whole story of power creep. Take my earlier statement discussing that a Pokemon with higher stats is more likely to win battles. I included a caveat in that statement about outside factors like abilities. it is nearly impossible to get away from said outside factors in a Pokemon game, and they will influence battles, making it so that a Pokemon with a lower BST can beat a Pokemon with a higher BST if the conditions are right. However, it is harder to quanitify power creep in these outside factors in an easily understandable numerical format and will require a lot more analysis, which will be subject to different outcomes depending on how someone chooses to measure power creep in these avenues. Though analyzing these different factors for power creep is still valid, using BST makes for a relatively simple, unbiased measure.

### What Pokemon will be included?

All Pokemon will be included in the analysis, excluding Mega Pokemon and Gigantimax forms of Pokemon as these are not separate species, simply a different form only accessible in battle. Pokemon of the same species with different forms (ex: Trash, Plant, and Sand Cloak Wormadam) will not have their forms counted separately unless there are differences in BSTs (their individual stats may switch around depending on the form, but their overall totals are the same). I will also examine BSTs by categories of Pokemon (standard, legendary, mythical), which will exclude special categories of Pokemon introduced in one generation, such as the Ultra Beasts from Generation 7, or the Paradox Pokemon from Generation 9. Baby Pokemon will also be excluded in this analysis. The three categories of Pokemon identified are used because there are new Pokemon introduced in all three categories in every generation, so there will be data over time to compare. 

Pokemon not included in base generation games but introduced later (ex: in deluxe re-releases, in DLC) will be included.

### What BSTs will be used from Pokemon in multiple generations?

Generally, BSTs stay the same for Pokemon between generations, so the stats for the most current generation will work. However, there are Pokemon who have had their stats changed between generations. These Pokemon will have their BSTs from their original generation used in the calculation, as these were the stats balanced for the generation they were released in. 

Additionally, from Generation 1 to Generation 2, the Special stat became two different stats: Special Attack and Special Defense, often with different combined values than their original special stat. As such, for all Generation 1 Pokemon, their Generation 2 BSTs will be used.

## Where is the data coming from?

All data being collected is from PokeAPI (link to PokeAPI). They offer many different APIs for collecting different data on Pokemon, and three will be used: their Generation API, their Pokemon Species API, and their Pokemon API.

Suppelementary data, such as the list of Pokemon who have had their stats changed between Generations, will come from Bulbapedia (link to Bulbapedia)


## Getting a list of all Pokemon

First, I am going to use PokeAPI's Generation API to collect a list of all of the Pokemon species introduced in every generation. A species, as defined by PokeAPI, "forms the basis for at least one Pokemon." Take the earlier Wormadam example and its three different forms. "Wormadam" is a species. Trash Cloak Wormadam is a form under the species of Wormadam. From this API, I will collect a list of API calls for the Pokemon Species API.

In [9]:
api_calls = []
name = []
for i in range(1,10):
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='w') as f:
        request = requests.get(f"https://pokeapi.co/api/v2/generation/{i}")
        pokemon_json = request.json()
        json.dump(pokemon_json,f,indent=4)
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path= 'pokemon_species'
        )
    for i in range(len(df['url'])):
        api_calls.append(df['url'][i])
        name.append(df['name'][i])

api_calls[0]


'https://pokeapi.co/api/v2/pokemon-species/1/'

Now that I have a list of all the different Pokemon species' APIs, I am going to call each API individually to get a list of all of the Pokemons' IDs, names, is_baby, is_legendary, and is_mythical classifications, and varieties.

### Bringing in all of the Pokemon data

In [43]:
for i in range(len(api_calls)):
        with open (f'../data/raw/{name[i]}_spec.json', mode='w') as f:
                request = requests.get(api_calls[i])
                pokemon_json = request.json()
                json.dump(pokemon_json,f,indent=4)

.rename(): I realized that when I was creating the separate data frame for the Pokemon's generation data, some of the column names were being repeated (ex: 'name' being used for both the name of the Pokemon species and the name of the generation where the Pokemon originated from). As such, I decided to use the .rename function, which takes a dictionary using the original names as keys and the new names as the values. You can rename either columns or indexes with these, but I chose to rename columns to eliminate duplicate names.

In [ ]:
main_df = ''
with open(f'../data/raw/{name[0]}_spec.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path = 'varieties',
            meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical', 'forms_switchable'],
            sep='_'
        )
        gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})

initial_df = pd.concat([df,gen_df],axis=1)


for i in range(1, len(api_calls)):
        with open(f'../data/raw/{name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical', 'forms_switchable'],
                sep='_'
                )
                gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})
        dfc = pd.concat([df,gen_df],axis=1)
        main_df = pd.concat([dfc,initial_df],axis=0, ignore_index = True)
        initial_df = main_df

stat_calls = []
form_name = []

for i in range(len(main_df['pokemon_url'])):
        stat_calls.append(main_df['pokemon_url'][i])
        form_name.append(main_df['pokemon_name'][i])



1351
1351
1351


### Retrieving Pokemon base stat data

In [59]:
for i in range(len(stat_calls)):
        with open (f'../data/raw/{form_name[i]}_spec.json', mode='w') as f:
                request = requests.get(stat_calls[i])
                pokemon_json = request.json()
                json.dump(pokemon_json,f,indent=4)